# Notebook 3: Structure Prediction with Boltz2 on the HPC

**Pipeline step:** Predict 3D structures of all 17 cupin and 17 MetRS domain sequences using Boltz2, running on the Hábrók GPU cluster via SLURM.

**Overview:** This notebook explains how protein 3D structure prediction works, what Boltz2 is and how it is configured, and how to run large computational jobs on an HPC cluster using the SLURM job scheduler. We walk through every line of the SLURM submission script used in AAR-COMP-012.

---

## Learning objectives

- Understand protein structure prediction and its recent history
- Know what PDB/CIF files contain and how to read them with Biopython
- Understand what Boltz2 is and how its YAML input files are structured
- Understand what an HPC cluster is, what nodes and GPUs are
- Know what SLURM is and why it is needed
- Read and interpret every SLURM directive in the submission script
- Submit a job, monitor it, and interpret the output

---

## 1. Why predict protein structures?

A protein's 3D structure determines its function. Knowing which residues line the active site, how the zinc ion is coordinated, and how the substrate fits — all of this requires 3D information.

Experimental structure determination is expensive and slow:
- **X-ray crystallography** — the gold standard; requires growing diffraction-quality crystals, which often fails
- **NMR spectroscopy** — limited to smaller proteins (< ~50 kDa) in solution
- **Cryo-EM** — powerful for large complexes; still requires purified protein

All three methods take months to years per protein. Computational structure prediction has transformed the field:

| Year | Method | Key advance |
|------|--------|-------------|
| 1990s | Homology modelling | Copy structure from a related solved protein |
| 2020 | AlphaFold2 | Deep learning on evolutionary covariation; near-experimental accuracy |
| 2023 | ESMFold, RoseTTAFold2 | Fast, competitive models |
| 2024 | Boltz2 | Diffusion model; handles proteins, ligands, nucleic acids jointly |

> **Key concept:** Boltz2 is not just a structure predictor — it is a generative diffusion model that samples from a probability distribution over 3D structures. Multiple samples (here: 3 per sequence) give an ensemble and a confidence estimate.

---

## 2. What is a CIF file?

Boltz2 outputs structures in **mmCIF format** (`.cif`). CIF stands for Crystallographic Information File. It is the modern standard, replacing the older PDB format. A CIF file is a plain-text table with columns for:

- `_atom_site.label_seq_id` — residue sequence number
- `_atom_site.label_comp_id` — residue name (e.g. `SER`, `HIS`, `ZN`)
- `_atom_site.label_atom_id` — atom name (e.g. `CA`, `N`, `O`)
- `_atom_site.Cartn_x/y/z` — Cartesian coordinates in Ångströms
- `_atom_site.B_iso_or_equiv` — B-factor (temperature factor; Boltz2 uses this to report pLDDT confidence per residue)

Biopython can read CIF files with `MMCIFParser`.

In [ ]:
# ── Biopython structure hierarchy demo ────────────────────────────────
# Biopython organises a structure as a 4-level hierarchy:
#   Structure → Model → Chain → Residue → Atom

from Bio.PDB import PDBParser

# Load PyrN.pdb (X-ray crystal structure — reference for MetRS analysis)
PYRN_PDB = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step4/PyrN.pdb"

parser   = PDBParser(QUIET=True)   # QUIET=True suppresses warnings
structure = parser.get_structure("PyrN", PYRN_PDB)

# Navigate the hierarchy
model  = structure[0]      # first MODEL (most PDB files have one)
chain  = model["A"]        # chain A

# Count residues (skip HETATM records like water and ZN)
residues = [r for r in chain if r.id[0] == " "]

print("PyrN crystal structure:")
print("  Total residues (chain A): {}".format(len(residues)))
print("  First residue: {}  (res id: {})".format(
    residues[0].resname, residues[0].id[1]))
print("  Last  residue: {}  (res id: {})".format(
    residues[-1].resname, residues[-1].id[1]))

In [ ]:
# ── Look at one residue in detail ─────────────────────────────────────
# Pick residue 138 (S138, the first AMP-site residue)
res = chain[138]    # access by residue number

print("Residue 138: {}".format(res.resname))  # should be SER
print("Atoms:")
for atom in res:
    # atom.get_vector() returns the 3D coordinates
    vec = atom.get_vector()
    print("  {:4s}  x={:7.3f}  y={:7.3f}  z={:7.3f}".format(
        atom.name, vec[0], vec[1], vec[2]))

---

## 3. Boltz2 and YAML input format

**Boltz2** (Boltz v2) is an open-source diffusion model developed at MIT for biomolecular structure prediction. Inputs are YAML files — one per structure to predict.

**YAML** (Yet Another Markup Language) is a human-readable format for structured data. Think of it as a recipe card: ingredient names on the left, values on the right, indentation shows nesting.

```yaml
sequences:           # top-level key
  - protein:         # first item: a protein chain
      id: A          # chain identifier
      sequence: MTVP... # amino acid sequence
  - ligand:          # second item: a small molecule
      id: ZN1        # ligand identifier
      ccd: ZN        # CCD code for zinc ion
```

> **Important:** Indentation in YAML is not decorative — it is part of the syntax. Two spaces under `protein:` means those keys belong to that protein block. Wrong indentation causes a parse error.

The cupin YAML includes 1 ZN ligand; the MetRS YAML includes 2 ZN ligands (matching the known cofactor stoichiometries).

In [ ]:
# ── Examine an actual Boltz2 YAML from the pipeline ───────────────────
import glob

# Find one cupin YAML
yaml_files = glob.glob(
    "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/yaml/cupin_batch1/*.yaml"
)

if yaml_files:
    with open(yaml_files[0]) as fh:
        content = fh.read()
    print("File: {}".format(yaml_files[0].split("/")[-1]))
    print()
    print(content)
else:
    print("No YAML files found — run generate_yamls.py first.")

In [ ]:
# ── Walk through generate_yamls.py logic ──────────────────────────────
# The script uses a simple template with .format() substitution

CUPIN_TEMPLATE = """sequences:
  - protein:
      id: A
      sequence: {sequence}
  - ligand:
      id: ZN1
      ccd: ZN
"""

METRS_TEMPLATE = """sequences:
  - protein:
      id: A
      sequence: {sequence}
  - ligand:
      id: ZN1
      ccd: ZN
  - ligand:
      id: ZN2
      ccd: ZN
"""

# Demonstrate filling the template for PyrN cupin
example_seq = "PAWAEAAGVPFNVRRVTLRPGETTAEHNHHDLEVWVMLD"  # shortened for display
filled = CUPIN_TEMPLATE.format(sequence=example_seq)
print("Filled YAML:")
print(filled)

---

## 4. HPC clusters: what they are and why we need one

**HPC (High Performance Computing)** clusters are collections of many computers (called **nodes**) connected by a fast network. The University of Groningen's cluster is called **Hábrók**.

```
You (login node)  →  submit job  →  SLURM  →  compute node with GPU
```

Key concepts:

- **Login node:** The machine you connect to via SSH. Used for small tasks (editing, compiling, submitting jobs) but NOT for heavy computation.
- **Compute node:** A powerful machine with many CPUs, large RAM, and optionally a GPU. This is where Boltz2 runs.
- **GPU (Graphics Processing Unit):** Originally designed for rendering graphics, GPUs have thousands of small cores ideal for the matrix mathematics inside neural networks like Boltz2.
- **Scratch storage:** `/scratch/` is fast, temporary storage on Hábrók. Home directories (`~`) are backed up but slower and have small quotas. Always run computations on scratch.

> **Key concept:** Never run Boltz2 directly on the login node. It would slow the cluster for everyone and may be killed automatically. Always submit via `sbatch`.

---

## 5. SLURM: the job scheduler

**SLURM** (Simple Linux Utility for Resource Management) is the software that decides which job runs on which node and when. It is like a queue at a busy restaurant: you submit your order (job), SLURM puts it in the queue, and when a table (compute node) with the right resources is free, your job starts.

Key SLURM commands:

| Command | What it does |
|---|---|
| `sbatch script.sh` | Submit a job script to the queue |
| `squeue -u $USER` | Show your running/pending jobs |
| `scancel <JOBID>` | Cancel a job |
| `seff <JOBID>` | After completion: show CPU/memory efficiency |

SLURM reads resource requests from `#SBATCH` comment lines at the top of the shell script.

---

## 6. The SLURM script — line by line

Here is `AAR-012-Step3-cupin1.sh` with every line explained.

In [ ]:
# ── Read and display the SLURM script ─────────────────────────────────
SLURM_SCRIPT = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/AAR-012-Step3-cupin1.sh"

with open(SLURM_SCRIPT) as fh:
    print(fh.read())

**Line-by-line explanation:**

```bash
#!/bin/bash                         # Shebang: tells the OS to run this with bash
#SBATCH --job-name=AAR-012-cupin1   # Name shown in squeue
#SBATCH --partition=gpu             # Which queue to use (gpu = nodes with GPUs)
#SBATCH --cpus-per-task=4           # CPU cores reserved for data loading etc.
#SBATCH --mem=80G                   # RAM requested (80 GB is safe for Boltz2)
#SBATCH --time=24:00:00             # Maximum walltime (HH:MM:SS); job is killed if exceeded
#SBATCH --nodes=1                   # Use exactly 1 compute node
#SBATCH --ntasks=1                  # Run 1 task (1 Boltz2 process)
#SBATCH --gres=gpu:1                # Request 1 GPU (generic resource)
#SBATCH --output=...cupin1_%j.log   # Stdout log file; %j is replaced by the job ID
#SBATCH --error=...cupin1_%j.err    # Stderr log file
```

```bash
set -euo pipefail   # Safer bash: exit on any error, undefined variable, or pipe failure
```

```bash
module purge            # Remove all currently loaded modules (start clean)
module load CUDA/12.1.1 # Load CUDA drivers so PyTorch can use the GPU
```

```bash
source .../conda.sh     # Initialise conda in this bash session
conda activate boltz2   # Activate the boltz2 conda environment
```

```bash
srun boltz predict "$YAML_DIR" \   # srun runs the job within SLURM's resource allocation
    --cache "$CACHE_DIR" \          # Where Boltz2 stores model weights (download once)
    --out_dir "$OUT_DIR" \          # Where to write CIF outputs
    --use_msa_server \              # Fetch MSA from ColabFold server (better predictions)
    --recycling_steps 3 \           # Number of structure refinement cycles
    --diffusion_samples 3 \         # Generate 3 independent structure samples
    --use_potentials                # Apply physical energy terms to guide sampling
```

> **Key concept:** `--diffusion_samples 3` means Boltz2 generates 3 independent 3D structures for each input. We then pick the one with the highest confidence score for further analysis.

---

## 7. Submitting and monitoring jobs

```bash
# Submit all four batches (run these commands in a terminal on Hábrók)
cd /scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/

sbatch AAR-012-Step3-cupin1.sh    # cupin batch 1 (9 sequences)
sbatch AAR-012-Step3-cupin2.sh    # cupin batch 2 (8 sequences)
sbatch AAR-012-Step3-metrs1.sh    # MetRS batch 1 (9 sequences)
sbatch AAR-012-Step3-metrs2.sh    # MetRS batch 2 (8 sequences)
```

Each `sbatch` command prints a job ID:
```
Submitted batch job 4512837
```

Monitor your jobs:
```bash
squeue -u $USER
```

Typical output:
```
 JOBID   PARTITION  NAME               USER     ST   TIME  NODES  NODELIST
4512837  gpu        AAR-012-cupin1     p318738  R    0:14  1      gpu001
4512838  gpu        AAR-012-cupin2     p318738  PD   0:00  1      (None)
```
`R` = running, `PD` = pending (waiting for resources)

Check the log file while the job is running:
```bash
tail -f /scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/logs/cupin1_4512837.log
```

---

## 8. Boltz2 output structure

For a batch of YAMLs named `Spb40_cupin.yaml`, `PyrN_cupin.yaml`, etc., Boltz2 creates:

```
cupin/
  boltz_results_cupin_batch1/
    predictions/
      Spb40_cupin/
        Spb40_cupin_model_0.cif          # structure sample 0
        Spb40_cupin_model_1.cif          # structure sample 1
        Spb40_cupin_model_2.cif          # structure sample 2
        confidence_Spb40_cupin_model_0.json  # confidence metrics for sample 0
        confidence_Spb40_cupin_model_1.json
        confidence_Spb40_cupin_model_2.json
      PyrN_cupin/
        ...
```

In [ ]:
# ── Read a confidence JSON file ────────────────────────────────────────
import json
import glob

# Find all confidence JSON files for any enzyme in the cupin predictions
conf_files = sorted(glob.glob(
    "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/cupin/"
    "boltz_results_*/predictions/*/confidence_*.json"
))

if conf_files:
    # Show the first three confidence files found
    for cf in conf_files[:3]:
        with open(cf) as fh:
            data = json.load(fh)
        enzyme = cf.split("/")[-2]   # directory name = enzyme name
        model  = cf.split("_model_")[-1].replace(".json", "")
        print("Enzyme: {}  |  model_{}".format(enzyme, model))
        # Print all keys and values
        for key, val in data.items():
            if isinstance(val, (int, float, str)):
                print("  {:30s} {}".format(key, val))
        print()
else:
    print("No confidence JSON files found.")
    print("Run the SLURM jobs first, then re-run this cell.")

In [ ]:
# ── Find the best model for one enzyme ────────────────────────────────
# Replicate the logic of find_best_model() from analyse_metrs.py

from pathlib import Path

def find_best_model(pred_dir):
    """Return (cif_path, model_idx, confidence_score) for the best of 3 models."""
    best_score = -1    # start with an impossible low score
    best_cif   = None
    best_idx   = -1

    pred_dir = Path(pred_dir)   # ensure it is a Path object
    name     = pred_dir.name    # directory name = enzyme name

    for i in range(3):          # Boltz2 generates 3 samples (0, 1, 2)
        # Build the expected file names
        conf_json = pred_dir / "confidence_{}_model_{}.json".format(name, i)
        cif       = pred_dir / "{}_model_{}.cif".format(name, i)

        if conf_json.exists() and cif.exists():   # only if both files are present
            score = json.loads(conf_json.read_text()).get("confidence_score", 0)
            if score > best_score:       # keep track of the highest score
                best_score = score
                best_cif   = cif
                best_idx   = i

    return best_cif, best_idx, best_score


# Test on PyrN cupin predictions if they exist
test_dirs = glob.glob(
    "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step3/cupin/"
    "boltz_results_*/predictions/PyrN_cupin"
)

if test_dirs:
    cif, idx, score = find_best_model(test_dirs[0])
    print("PyrN cupin best model:")
    print("  model_{} — confidence_score = {:.4f}".format(idx, score))
    print("  CIF path: {}".format(cif))
else:
    print("PyrN cupin predictions not found — run SLURM jobs first.")

---

## Summary

In this notebook you:
- Learned the history of protein structure prediction from X-ray crystallography to Boltz2
- Read a PDB file with Biopython and navigated the Structure-Model-Chain-Residue-Atom hierarchy
- Understood the YAML input format for Boltz2 (protein sequence + ZN ligand blocks)
- Learned what an HPC cluster is, what SLURM does, and how to submit, monitor, and cancel jobs
- Traced every line of the SLURM submission script used in AAR-COMP-012
- Saw how to parse Boltz2 confidence JSON files to select the best structural model

**Next:** Notebook 4 will take the predicted CIF files, align them to the PyrN reference structure, and extract active site residues.

---

## Try it yourself

**Exercise 1:** Read one of the Boltz2 confidence JSON files (once predictions are available) and print the `confidence_score` and `ptm` for all 3 models of one enzyme.

```python
# Hint: glob.glob() to find files, json.load() to read them
# your code here
```

**Exercise 2:** Write a YAML by hand for a made-up 10-residue protein sequence `MHVKTLDRWE` with one ZN ligand. Save it to `/tmp/test.yaml` and print it to verify the indentation is correct.

```python
# Hint: use CUPIN_TEMPLATE.format(sequence="MHVKTLDRWE")
# and write it to file with open('/tmp/test.yaml', 'w')
# your code here
```

**Exercise 3:** Write the `sbatch` command to submit the cupin batch 2 script, and write the `squeue` command to check its status afterwards.

```bash
# Write your answers here as comments:
# sbatch command:
# squeue command:
```